# Stage 5 · Per-major-type L2 clustering

Within each major type, integrate mCG and 3C (per-type refined anchors) and consensus-cluster into **L2any subtypes**. Example major type: **Epi-Ent**.

These are the original pipeline notebooks, concatenated in execution order with paths normalized to `ENTEX_ROOT`. They document the method and run per tissue / major type across the full raw dataset (heavy compute); they are shown for reference and are **not re-executed in the book**. Example group shown where templated.

In [1]:
# === Reproduction setup ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT); import repro_guard


## 5a · mCG subtype embedding + integration

_Source: `clustering/merged/L2/Epi-Ent/01.mc_embed_final.ipynb`_

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from glob import glob

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [3]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [4]:
group_name = 'Neu-Exc'

In [5]:
# Parameters
cpu = 8
group_name = "Epi-Ent"
mem_gb = 30


In [6]:
mcad = anndata.read_h5ad('5kCG_embed.h5ad')
mcad

In [7]:
npc = pd.read_csv(f'{ENTEX_ROOT}/clustering/merged/npc.tsv', sep='\t', header=None, index_col=0).loc[group_name, 1]
print(npc)

In [8]:
sample_list = np.sort(mcad.obs['Donor'].astype(str).unique())
sample_list

In [9]:
mcad.obs['Donor'].value_counts()

In [10]:
adata_list = [mcad[mcad.obs['Donor']==xx] for xx in sample_list]


In [11]:
anchor_index = {}
for xx,adata in zip(sample_list, adata_list):
    tmp = pd.DataFrame(index=adata.obs.index).reset_index().reset_index()
    anchor_index[xx] = tmp.set_index('cell')['index'].to_dict()


In [12]:
outdir = f'{ENTEX_ROOT}/clustering/'
ct_list = np.sort([xx.split('/')[-2] for xx in glob(f'{outdir}tissue/L2/*-*/mCG_5kb_seurat_anchor_*.hdf')])
print(len(ct_list))


In [13]:
anchor = []
for ct in ct_list:
    anchor_file = glob(f'{outdir}tissue/L2/{ct}/mCG_5kb_seurat_anchor_*.hdf')[0]
    adata_file = glob(f'{outdir}tissue/L2/{ct}/mCG_5kb_lsi.h5ad')[0]
    anchor_tmp = pd.read_hdf(anchor_file)
    adata_tmp = anndata.read_h5ad(adata_file)
    sample_tmp = np.sort(adata_tmp.obs['Donor'].unique())
    cell_list = [adata_tmp.obs.index[adata_tmp.obs['Donor']==xx] for xx in sample_tmp]
    thres = np.percentile(anchor_tmp['dist_pc'], 90)
    anchor_tmp = anchor_tmp.loc[anchor_tmp['dist_pc']<thres, ['x1','x2','score']].copy()
    anchor_tmp['x1'] = cell_list[0][anchor_tmp['x1']]
    anchor_tmp['x2'] = cell_list[1][anchor_tmp['x2']]
    anchor_tmp = anchor_tmp.loc[anchor_tmp['x1'].isin(mcad.obs.index) & anchor_tmp['x2'].isin(mcad.obs.index)]
    if anchor_tmp.shape[0]>0:
        anchor_tmp['x1'] = anchor_tmp['x1'].map(anchor_index[sample_tmp[0]])
        anchor_tmp['x2'] = anchor_tmp['x2'].map(anchor_index[sample_tmp[1]])
        anchor_tmp['x1_donor'] = sample_tmp[0]
        anchor_tmp['x2_donor'] = sample_tmp[1]
        anchor.append(anchor_tmp)
    
anchor = pd.concat(anchor, axis=0)


In [14]:
anchor[['x1_donor','x2_donor']].value_counts().sort_index()


In [15]:
n = len(adata_list)
anchor_df = {}

for i in range(n - 1):
    for j in range(i + 1, n):
        x1, x2 = sample_list[[i,j]]
        selc = (anchor['x1_donor']==x1) & (anchor['x2_donor']==x2)
        anchor_df[(i,j)] = anchor.loc[selc, ['x1', 'x2', 'score']]
        if selc.sum()==0:
            selc = (anchor['x1_donor']==x2) & (anchor['x2_donor']==x1)
            anchor_df[(i,j)] = anchor.rename({'x1':'x2','x2':'x1'}, axis=1).loc[selc, ['x1', 'x2', 'score']]
        

In [16]:
integrator = SeuratIntegration()


In [17]:
integrator.adata_dict = {k: v for k, v in zip(list(range(len(adata_list))), adata_list)}
integrator.n_dataset = len(adata_list)
integrator.n_cells = [adata.shape[0] for adata in adata_list]

# intra-dataset KNN for scoring the anchors
integrator.k_local = None
integrator.key_local = 'X_lsi'
integrator._calculate_local_knn()

# alignments and all_pairs
integrator.alignments = None
integrator._get_all_pairs()
integrator.anchor = anchor_df


In [18]:
corrected = integrator.integrate(key_correct='5kCG_lsi',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'5kCG_u{npc}_seuratL2'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'5kCG_u{npc}_seuratL2'] = normalize(mcad.obsm[f'5kCG_u{npc}_seuratL2'][:, :npc], axis=1)

tsne(mcad, obsm=f'5kCG_u{npc}_seuratL2', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=8)
mcad.obsm[f'5kCG_u{npc}_seuratL2_tsne'] = mcad.obsm['X_tsne'].copy()


In [19]:
mcad.write_h5ad('5kCG_embed.h5ad')

## 5b · 3C subtype embedding + integration

_Source: `clustering/merged/L2/Epi-Ent/02.hic_embed_final.ipynb`_

In [20]:
import numpy as np
import pandas as pd
from scipy.stats import zscore, pearsonr
from glob import glob

from sklearn.preprocessing import normalize, OneHotEncoder
from sklearn.decomposition import TruncatedSVD

import anndata
import scanpy as sc
import scanpy.external as sce

from ALLCools.clustering import *
from ALLCools.integration.seurat_class import SeuratIntegration

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

from ALLCools.plot import *

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'
 

In [21]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [22]:
group_name = 'Neu-Exc'

In [23]:
# Parameters
cpu = 8
group_name = "Epi-Ent"
mem_gb = 30


In [24]:
indir = f'{ENTEX_ROOT}/clustering/'


In [25]:
chromsize = pd.read_csv(f'{REF_ROOT}/hg38/fasta/hg38.main.chrom.sizes', sep='\t', index_col=0, header=None)
chromsize = chromsize.iloc[:22]


In [26]:
mcad = anndata.read_h5ad('100k3C_embed.h5ad')
mcad

In [27]:
npc = pd.read_csv(f'{ENTEX_ROOT}/clustering/merged/npc.tsv', sep='\t', header=None, index_col=0).loc[group_name, 2]
print(npc)

In [28]:
sample_list = np.sort(mcad.obs['Donor'].astype(str).unique())
sample_list

In [29]:
mcad.obs['Donor'].value_counts()

In [30]:
adata_list = [mcad[mcad.obs['Donor']==xx] for xx in sample_list]


In [31]:
anchor_index = {}
for xx,adata in zip(sample_list, adata_list):
    tmp = pd.DataFrame(index=adata.obs.index).reset_index().reset_index()
    anchor_index[xx] = tmp.set_index('cell')['index'].to_dict()


In [32]:
ct_list = np.sort([xx.split('/')[-2] for xx in glob(f'{indir}tissue/L2/*-*/HiC_100kb_seurat_anchor_*.hdf')])
print(len(ct_list))


In [33]:
anchor = []
for ct in ct_list:
    anchor_file = glob(f'{indir}tissue/L2/{ct}/HiC_100kb_seurat_anchor_*.hdf')[0]
    adata_file = glob(f'{indir}tissue/L2/{ct}/HiC_100kb_pca.h5ad')[0]
    anchor_tmp = pd.read_hdf(anchor_file)
    adata_tmp = anndata.read_h5ad(adata_file)
    sample_tmp = np.sort(adata_tmp.obs['Donor'].unique())
    cell_list = [adata_tmp.obs.index[adata_tmp.obs['Donor']==xx] for xx in sample_tmp]
    thres = np.percentile(anchor_tmp['dist_pc'], 90)
    anchor_tmp = anchor_tmp.loc[anchor_tmp['dist_pc']<thres, ['x1','x2','score']].copy()
    anchor_tmp['x1'] = cell_list[0][anchor_tmp['x1']]
    anchor_tmp['x2'] = cell_list[1][anchor_tmp['x2']]
    anchor_tmp = anchor_tmp.loc[anchor_tmp['x1'].isin(mcad.obs.index) & anchor_tmp['x2'].isin(mcad.obs.index)]
    if anchor_tmp.shape[0]>0:
        anchor_tmp['x1'] = anchor_tmp['x1'].map(anchor_index[sample_tmp[0]])
        anchor_tmp['x2'] = anchor_tmp['x2'].map(anchor_index[sample_tmp[1]])
        anchor_tmp['x1_donor'] = sample_tmp[0]
        anchor_tmp['x2_donor'] = sample_tmp[1]
        anchor.append(anchor_tmp)
    
anchor = pd.concat(anchor, axis=0)


In [34]:
anchor[['x1_donor','x2_donor']].value_counts().sort_index()


In [35]:
n = len(adata_list)
anchor_df = {}

for i in range(n - 1):
    for j in range(i + 1, n):
        x1, x2 = sample_list[[i,j]]
        selc = (anchor['x1_donor']==x1) & (anchor['x2_donor']==x2)
        anchor_df[(i,j)] = anchor.loc[selc, ['x1', 'x2', 'score']]
        if selc.sum()==0:
            selc = (anchor['x1_donor']==x2) & (anchor['x2_donor']==x1)
            anchor_df[(i,j)] = anchor.rename({'x1':'x2','x2':'x1'}, axis=1).loc[selc, ['x1', 'x2', 'score']]
        

In [36]:
integrator = SeuratIntegration()


In [37]:
integrator.adata_dict = {k: v for k, v in zip(list(range(len(adata_list))), adata_list)}
integrator.n_dataset = len(adata_list)
integrator.n_cells = [adata.shape[0] for adata in adata_list]

# intra-dataset KNN for scoring the anchors
integrator.k_local = None
integrator.key_local = 'X_pca'
integrator._calculate_local_knn()

# alignments and all_pairs
integrator.alignments = None
integrator._get_all_pairs()
integrator.anchor = anchor_df


In [38]:
corrected = integrator.integrate(key_correct='100k3C_pca',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'100k3C_pc{npc}_seuratL2'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'100k3C_pc{npc}_seuratL2'] = normalize(mcad.obsm[f'100k3C_pc{npc}_seuratL2'][:, :npc], axis=1)

tsne(mcad, obsm=f'100k3C_pc{npc}_seuratL2', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=8)
mcad.obsm[f'100k3C_pc{npc}_seuratL2_tsne'] = mcad.obsm['X_tsne'].copy()


In [39]:
mcad.write_h5ad('100k3C_embed.h5ad')


## 5c · joint embedding → L2any subtypes

_Source: `clustering/merged/L2/Epi-Ent/03.joint_embed_final.ipynb`_

In [40]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [41]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [42]:
group_name = 'Neu-Exc'

In [43]:
# Parameters
cpu = 8
group_name = "Epi-Ent"
mem_gb = 30


In [44]:
adata_mc = anndata.read_h5ad('5kCG_embed.h5ad')
adata_3c = anndata.read_h5ad('100k3C_embed.h5ad')
adata_3c = adata_3c[adata_mc.obs.index].copy()


In [45]:
# npc_cg = [int(xx.split('_')[1][1:]) for xx in adata_mc.obsm.keys() if '_seuratL2_tsne' in xx][0]
# # npc_3c = [int(xx.split('_')[1][1:]) for xx in adata_3c.obsm.keys() if '100k3C_u' in xx][0]
# npc_3c = [int(xx.split('_')[1][2:]) for xx in adata_3c.obsm.keys() if '_seuratL2_tsne' in xx][0]
npc_cg, npc_3c = pd.read_csv(f'{ENTEX_ROOT}/clustering/merged/npc.tsv', sep='\t', header=None, index_col=0).loc[group_name].values
print(npc_cg, npc_3c)


In [46]:
adata = anndata.AnnData(obs=adata_mc.obs)
# adata.obsm[f'5kCG100k3C_u{npc_cg}u{npc_3c}'] = np.hstack([normalize(adata_mc.obsm[f'5kCG_u{npc_cg}hm'], axis=1),
#                                                           normalize(adata_3c.obsm['X_pca'], axis=1)])
adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}'] = np.hstack([normalize(adata_mc.obsm[f'5kCG_u{npc_cg}_seuratL2'], axis=1),
                                                           normalize(adata_3c.obsm[f'100k3C_pc{npc_3c}_seuratL2'], axis=1)])
tsne(adata, obsm=f'5kCG100k3C_u{npc_cg}pc{npc_3c}', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=8)
adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}_tsne'] = adata.obsm['X_tsne'].copy()


In [47]:
ds = 2
coord_base = 'tsne'
adata.obsm[f'X_{coord_base}'] = adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}_{coord_base}'].copy()
dump_embedding(adata, coord_base)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300, constrained_layout=True)

tmp = adata.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        # palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True
                       )

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Tissue',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True
                       )

# ax = axes[0, 2]
# _ = categorical_scatter(data=tmp,
#                         ax=ax,
#                         coord_base=coord_base,
#                         hue='leiden_cons',
#                         text_anno='leiden_cons', 
#                         s=ds,
#                         labelsize=8,
#                         max_points=None,
#                         palette='tab20',
#                         scatter_kws={'rasterized':True},
#                         # legend_kws={'ncol':1}, 
#                         # show_legend=True
#                        )

ax = axes[0, 2]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )


ax = axes[1, 0]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCGFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 2]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCHFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )



In [48]:
# clustering name
clustering_name = 'L1'

# ConsensusClustering
# Important factores
n_neighbors = 25
leiden_resolution = 1.0
# this parameter is the final target that limit the total number of clusters
# Higher accuracy means more conservative clustering results and less number of clusters
target_accuracy = 0.9
min_cluster_size = 30

# Other ConsensusClustering parameters
metric = 'euclidean'
consensus_rate = 0.8
leiden_repeats = 500
random_state = 0
train_frac = 0.5
train_max_n = 500
max_iter = 50
n_jobs = 8

# Dendrogram via Multiscale Bootstrap Resampling
nboot = 10000
method_dist = 'correlation'
method_hclust = 'average'

plot_type = 'static'

In [49]:
cc = ConsensusClustering(model=None,
                         n_neighbors=n_neighbors,
                         metric=metric,
                         min_cluster_size=min_cluster_size,
                         leiden_repeats=leiden_repeats,
                         leiden_resolution=leiden_resolution,
                         consensus_rate=consensus_rate,
                         random_state=random_state,
                         train_frac=train_frac,
                         train_max_n=train_max_n,
                         max_iter=max_iter,
                         n_jobs=n_jobs,
                         target_accuracy=target_accuracy)


In [50]:
cc.fit_predict(adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}'])

In [51]:
adata.obs['leiden_cons'] = cc.label.copy()
adata.obs['leiden_cv'] = cc.cv_predicted_label.copy()


In [52]:
cc.save('ConcensusClustering.model.lib')
adata.write_h5ad('5kCG100k3C_embed.h5ad')


In [53]:
from sklearn.metrics import adjusted_rand_score as ARI
ARI(adata.obs['leiden_cons'], adata.obs['leiden_cv'])


In [54]:
if len(cc.step_data)>1:
    cc.target_accuracy = 0
    cc.supervise_learning()
    cc.final_evaluation()
    adata.obs['leiden_init'] = cc.label.copy()
else:
    adata.obs['leiden_init'] = adata.obs['leiden_cons'].copy()
    

In [55]:
ds = 2
coord_base = 'tsne'
adata.obsm[f'X_{coord_base}'] = adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}_{coord_base}'].copy()
dump_embedding(adata, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = adata.obs.copy()
ax = axes[0,0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='leiden_cons',
                        text_anno='leiden_cons',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[0,1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='leiden_cv',
                        text_anno='leiden_cv',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[1,0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Tissue',
                        # text_anno='celltype', 
                        s=ds,
                        labelsize=6,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True
                       )

ax = axes[1,1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=6,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )

# plt.savefig(f'{group_name}_5kCG100k3C_cluster.pdf', transparent=True)


In [56]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12), dpi=300, constrained_layout=True)

for i,xx in enumerate([adata, adata_mc, adata_3c]):
    dump_embedding(xx, coord_base)
    tmp = xx.obs.copy()
    tmp['leiden_init'] = adata.obs['leiden_init'].copy()
    tmp['ClusterTissue'] = adata.obs['ClusterTissue'].copy()
    ax = axes[0, i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='Donor',
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            # palette='tab20',
                            scatter_kws={'rasterized':True},
                            show_legend=True)

    ax = axes[1, i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='leiden_init',
                            text_anno='leiden_init', 
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            legend_kws={'ncol':1}, 
                            show_legend=True)

    ax = axes[2, i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='ClusterTissue',
                            text_anno='ClusterTissue', 
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1}, 
                            # show_legend=False
                           )


## 5d · consensus subtype clustering (acc90)

_Source: `clustering/merged/L2/Epi-Ent/04.clustering.ipynb`_

In [57]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [58]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [59]:
group_name = 'Neu-Exc'

In [60]:
# Parameters
cpu = 4
group_name = "Epi-Ent"
mem_gb = 16


In [61]:
adata_mc = anndata.read_h5ad('5kCG_embed.h5ad')
adata_3c = anndata.read_h5ad('100k3C_embed.h5ad')
adata_3c = adata_3c[adata_mc.obs.index].copy()


In [62]:
npc_cg = [int(xx.split('_')[1][1:]) for xx in adata_mc.obsm.keys() if 'seuratfilter60_tsne' in xx][0]
# npc_3c = [int(xx.split('_')[1][1:]) for xx in adata_3c.obsm.keys() if '100k3C_u' in xx][0]
npc_3c = [int(xx.split('_')[1][2:]) for xx in adata_3c.obsm.keys() if 'seuratfilter60_tsne' in xx][0]
print(npc_cg, npc_3c)


In [63]:
adata = anndata.read_h5ad('5kCG100k3C_embed.h5ad')
adata

In [64]:
# clustering name
clustering_name = 'L1'

# ConsensusClustering
# Important factores
n_neighbors = 25
leiden_resolution = 1.0
# this parameter is the final target that limit the total number of clusters
# Higher accuracy means more conservative clustering results and less number of clusters
target_accuracy = 0.9
min_cluster_size = 30

# Other ConsensusClustering parameters
metric = 'euclidean'
consensus_rate = 0.8
leiden_repeats = 500
random_state = 0
train_frac = 0.5
train_max_n = 500
max_iter = 50
n_jobs = 4

# Dendrogram via Multiscale Bootstrap Resampling
nboot = 10000
method_dist = 'correlation'
method_hclust = 'average'

plot_type = 'static'

In [65]:
cc = ConsensusClustering(model=None,
                         n_neighbors=n_neighbors,
                         metric=metric,
                         min_cluster_size=min_cluster_size,
                         leiden_repeats=leiden_repeats,
                         leiden_resolution=leiden_resolution,
                         consensus_rate=consensus_rate,
                         random_state=random_state,
                         train_frac=train_frac,
                         train_max_n=train_max_n,
                         max_iter=max_iter,
                         n_jobs=n_jobs,
                         target_accuracy=target_accuracy)


In [66]:
cc.fit_predict(adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}'])

In [67]:
adata.obs['leiden_cons_acc90'] = cc.label.copy()
adata.obs['leiden_cv_acc90'] = cc.cv_predicted_label.copy()


In [68]:
cc.save('ConcensusClustering_acc90.model.lib')
adata.write_h5ad('5kCG100k3C_embed.h5ad')


In [69]:
from sklearn.metrics import adjusted_rand_score as ARI
ARI(adata.obs['leiden_cons_acc90'], adata.obs['leiden_cv_acc90'])


In [70]:
ds = 2
coord_base = 'tsne'
adata.obsm[f'X_{coord_base}'] = adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}_{coord_base}'].copy()
dump_embedding(adata, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = adata.obs.copy()
ax = axes[0,0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='leiden_cons_acc90',
                        text_anno='leiden_cons_acc90',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[0,1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='leiden_cv_acc90',
                        text_anno='leiden_cv_acc90',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[1,0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Tissue',
                        # text_anno='celltype', 
                        s=ds,
                        labelsize=6,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True
                       )

ax = axes[1,1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=6,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )

# plt.savefig(f'{group_name}_5kCG100k3C_cluster.pdf', transparent=True)


In [71]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300, constrained_layout=True)

for i,xx in enumerate([adata, adata_mc, adata_3c]):
    dump_embedding(xx, coord_base)
    tmp = xx.obs.copy()
    tmp['leiden_cons'] = adata.obs['leiden_cons_acc90'].copy()
    ax = axes[0, i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='Donor',
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            show_legend=True)

    ax = axes[1, i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='leiden_cons',
                            text_anno='leiden_cons', 
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            legend_kws={'ncol':1}, 
                            show_legend=True)


In [72]:
selc = [[0], []
       ]
adata_mc.obs['leiden_merge'] = '0'
for k,xx in enumerate(selc):
    selcell = adata_mc.obs['leiden'].astype(int).isin(xx)
    adata_mc.obs.loc[selcell, 'leiden_merge'] = str(k)
    

In [73]:
ds = 2
coord_base = 'tsne'
fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300, constrained_layout=True)

for i,xx in enumerate([adata, adata_mc, adata_3c]):
    dump_embedding(xx, coord_base)
    tmp = xx.obs.copy()
    tmp['leiden_cons'] = adata.obs['leiden_cons_acc90'].copy()
    ax = axes[0, i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='Donor',
                            s=ds,
                            labelsize=12,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            show_legend=True)

    ax = axes[1, i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='leiden_cons',
                            text_anno='leiden_cons', 
                            s=ds,
                            labelsize=12,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            legend_kws={'ncol':1}, 
                            show_legend=True)
